# Análisis completo — flota anchovetera

Este notebook usa **exclusivamente** `data/raw/MapProbabilidad_adulto.csv`. No existe fallback a una grilla demo o sintética.

Las rutas, asignaciones y movimiento de los barcos son resultados/simulaciones del modelo.

In [ ]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd
from IPython.display import Image, display, HTML

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = ROOT / 'outputs'
DATA = ROOT / 'data' / 'raw' / 'MapProbabilidad_adulto.csv'
print('ROOT:', ROOT.resolve())
print('Dataset:', DATA.resolve())

## 1. Verificar dataset y ejecutar pipeline

In [ ]:
subprocess.check_call([sys.executable, str(ROOT/'scripts'/'materializar_datos.py')])
assert DATA.exists(), 'Falta MapProbabilidad_adulto.csv'
prob = pd.read_csv(DATA)
display(prob.head())
print('Filas totales:', len(prob), '| Prob válidas:', prob['Prob'].notna().sum())

In [ ]:
subprocess.check_call([
    sys.executable, str(ROOT/'scripts'/'run_pipeline.py'),
    '--sernanp', 'auto',
    '--precio-combustible', '5.0',
    '--costo-operativo-hora', '0.0'
])

## 2. Auditoría

In [ ]:
audit = json.loads((OUT/'auditoria.json').read_text(encoding='utf-8'))
audit

## 3. Mapa de probabilidad y zonas candidatas

In [ ]:
display(pd.read_csv(OUT/'zonas_candidatas.csv'))
display(Image(filename=str(OUT/'mapa_probabilidad_zonas.png')))

## 4. Comparación Greedy, MILP y MARL

In [ ]:
resumen = pd.read_csv(OUT/'resumen_metodos.csv')
display(resumen)
display(Image(filename=str(OUT/'grafico_comparacion_metodos.png')))

## 5. Asignaciones y mapas por método

In [ ]:
for metodo, csv, img in [
    ('Greedy','asignacion_greedy.csv','mapa_asignacion_greedy.png'),
    ('MILP','asignacion_milp.csv','mapa_asignacion_milp.png'),
    ('MARL','asignacion_marl.csv','mapa_asignacion_marl.png'),
]:
    print('\n', metodo)
    display(pd.read_csv(OUT/csv))
    display(Image(filename=str(OUT/img)))

## 6. Entrenamiento MARL sobre la grilla real

In [ ]:
hist = pd.read_csv(OUT/'historial_entrenamiento_marl.csv')
display(hist.tail(20))

## 7. Costos

In [ ]:
display(pd.read_csv(OUT/'costos_por_barco.csv'))
display(pd.read_csv(OUT/'costos_por_puerto.csv'))
display(Image(filename=str(OUT/'grafico_costos_por_barco.png')))
display(Image(filename=str(OUT/'grafico_combustible_por_puerto.png')))

## 8. Simulación MARL

La animación representa el movimiento **simulado** del plan generado por la política MARL; no es una trayectoria histórica.

In [ ]:
display(Image(filename=str(OUT/'simulacion_flota_marl.gif')))

## 9. Reporte HTML

In [ ]:
print((OUT/'reporte_resultados.html').resolve())
display(HTML('<b>Abre outputs/reporte_resultados.html para revisar tablas y gráficos juntos.</b>'))